# Alignment Bundles

An `AlignmentBundle` manages multiple timelines and the groups that
connect them. We load the Vienna 1x22 corpus (one score, 22
performances) using the `MatchfileLoader` to illustrate the pattern.

In [1]:
from timetoalign import MatchfileLoader
from timetoalign.testdata import ensure_data

DATA_DIR = ensure_data("vienna_1x22")

/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Load All Match Files at Once

A single `MatchfileLoader` instance processes all `.match` files that
share the same score.

In [2]:
match_files = sorted(DATA_DIR.glob("*.match"))
loader = MatchfileLoader()
loader.load(*match_files)

{
    "files loaded": len(match_files),
    "timelines": len(loader.create_timelines()),
}

{'files loaded': 22, 'timelines': 23}

## Create the Bundle

In [3]:
bundle = loader.create_bundle()
bundle

AlignmentBundle(id='bundle:AlignmentBundle_1', timelines=23, groups=1)

## Explore Timelines

In [4]:
score_tl = bundle.get_timeline("score")
score_tl

ContinuousLogicalTimeline(id='score:clt1', length=41.5, unit=quarters, events=454, children=0, cmaps=2)

In [5]:
score_tl.get_events(event_type="Note").to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration


## Query Coordinates Across Groups

`get_matchstamp_at()` is the cross-domain resolution method for an
`AlignmentBundle`: it accepts a coordinate on any timeline and returns a
`MatchStamp` spanning all connected timelines. `MatchStamp` shares the
unified stamp interface used by the other timestamp types.

In [6]:
perf_id = [uid for uid in bundle.timeline_ids if uid != score_tl.id][0]

stamp = bundle.get_matchstamp_at(100.0, "score:clt1")
stamp

ID,Coordinate,Type
score:clt1,100,


The returned stamp exposes unit-bearing coordinates through
`get_coordinate()`, and `is_interpolated` identifies a WarpMap fallback
rather than an exact claim anchor.

In [7]:
{
    "score coordinate": stamp.get_coordinate("score:clt1"),
    "interpolated": stamp.is_interpolated,
}

{'score coordinate': Coordinate(100.0, quarters), 'interpolated': True}

**Next:** [Flow Control and Grids](tut04_flow_and_grids.ipynb)